# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two Paper Findings + Methodology Questions

**Findings under audit:**
- **Finding #4** — The Freshness Multiplier ("365+ day content refreshed within 30 days shows 3.2x health boost and 57x more impressions")
- **Finding #1** — The Anatomy of Growing Content ("Growing content is 37.6% longer than declining content")

**Audit framework (per skeleton requirement):**
For each finding, we ask two questions:
1. **Label Provenance:** Where does the ground-truth label come from, and is it a trustworthy target?
2. **Validation Design:** Does the study design actually carry the causal or predictive weight the claim implies?

**Constructive tone commitment:** The FlyRank paper is unusually honest for an industry report — it flags its own unstable buckets (the 361+ cell with n=1 declining page), acknowledges survivor bias in the Age-Freshness Matrix, and explicitly warns against treating correlations as causal. Our audit builds on that honesty rather than dismissing the work. Where the paper's caveats are present but its headlines overshoot them, we propose tighter language. Where the empirical evidence can be independently tested on the starter dataset, we test it.

>  **Median reporting note:** All reported values below are independent column-level medians. `trend_pct` and `impressions_last_30d` are not derived from each other (median-of-ratios ≠ ratio-of-medians). Arithmetic reconciliation across columns is not expected.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# The file is directly in 'My Drive/content_refresh_anonymized.csv'
file_path = '/content/drive/My Drive/content_refresh_anonymized.csv'

### Finding #4: The Freshness Multiplier

#### 1A. Label Provenance

| Label / Metric | Source | Trustworthiness |
|---|---|---|
| "Refreshed within 30 days" | `days_since_update <= 30` | Observable timestamp. Clear, unambiguous. |
| "3.2x health boost" (10.7 → 34.5) | FlyRank Health Score composite | **Circularity concern.** Health Score = Impressions(30) + Position(30) + CTR(20) + Scroll(20). Since Impressions is 30% of the score, the "3.2x health boost" is partly *driven by* the same "57x impression" increase it is presented alongside. It is not an independent outcome measure. The paper itself acknowledges this circularity in the ML Appendix ("health score is partly constructed from inputs such as position and impressions"). |
| "57x more impressions" (71 → 4,039) | Raw GSC impressions, cross-sectional |  **Design concern.** This compares *different pages* (refreshed vs. not-refreshed) at one point in time, not the *same pages* before vs. after refresh. A cross-sectional snapshot cannot separate the refresh effect from pre-existing differences between the two groups. |
| Growth-to-decline ratio (361+ = 283:1) | Count of growing vs. declining pages in 361+ freshness bucket |  **Unstable.** The paper itself flags this: "283 growing pages versus only 1 declining." A ratio built on n=1 in the denominator carries no statistical weight. |

**Verdict on labels:** The "refreshed" timestamp is clean. The Health Score outcome is partially circular. The impression comparison is cross-sectional, not longitudinal. The 361+ ratio is unusable.

#### 1B. Validation Design Assessment

The paper's design for Finding #4 is:

Cross-sectional snapshot:
Group A = pages aged 365+ with days_since_update ≤ 30
Group B = pages aged 365+ with days_since_update > 30
Compare: mean impressions, mean health score


**What this design CAN show:** Association between recent refresh and higher current performance among old pages.

**What this design CANNOT show:**
- **Causality** — without a before/after on the same pages, we cannot know if refresh *caused* the improvement or if pages that were already performing better were simply more likely to be refreshed.
- **Magnitude** — the "57x" figure comes from comparing two groups that may differ in dozens of uncontrolled ways (topic, authority, historical traffic, client mix).
- **Generalizability** — the 361+ bucket contains only 1 declining page, making any ratio from it meaningless.

**What WOULD carry the claim:** A longitudinal before/after design on the same pages, or a matched-cohort design pairing refreshed pages with statistically similar non-refreshed pages and comparing their *change* over the same window.

In [12]:
# ============================================================
# FINDING #4: EMPIRICAL AUDIT ON STARTER DATASET
# ============================================================

old = df[df['content_age_days'] >= 365].copy()
old['recently_refreshed'] = old['days_since_last_update'] <= 30

n_total = len(old)
n_refreshed = int(old['recently_refreshed'].sum())
n_not = n_total - n_refreshed

print(f"=== STEP 1: Baseline Balance ===")
print(f"Old pages (365+): {n_total:,}")
print(f"  Refreshed: {n_refreshed:,} ({n_refreshed/n_total*100:.1f}%)")
print(f"  Not refreshed: {n_not:,} ({n_not/n_total*100:.1f}%)")
print(f"  → Refreshing is the operational default, not the exception.\n")

compare_cols = [c for c in ['impressions_prev_30d','word_count'] if c in old.columns]
print(old.groupby('recently_refreshed')[compare_cols].median())

print(f"\n=== STEP 2: Directional Outcome ===")
outcome_cols = [c for c in ['trend_pct','impressions_last_30d','impressions_90d'] if c in old.columns]
print(old.groupby('recently_refreshed')[outcome_cols].median())

print(f"\n=== STEP 3: Banded Check (controlling for baseline) ===")
imp_col = next((c for c in ['impressions_prev_30d','impressions_90d'] if c in old.columns), None)
if imp_col:
    old['baseline_band'] = pd.qcut(old[imp_col], q=4, duplicates='drop',
                                    labels=['Q1_Lowest','Q2','Q3','Q4_Highest'])
    banded = old.groupby(['baseline_band','recently_refreshed'],
                         observed=False)['trend_pct'].median().unstack()
    banded.columns = ['Not_Refreshed','Refreshed']
    banded['Gap'] = banded['Refreshed'] - banded['Not_Refreshed']
    print(banded.round(2))

=== STEP 1: Baseline Balance ===
Old pages (365+): 6,360
  Refreshed: 5,807 (91.3%)
  Not refreshed: 553 (8.7%)
  → Refreshing is the operational default, not the exception.

                    impressions_prev_30d  word_count
recently_refreshed                                  
False                              319.0      3818.0
True                               240.0      2820.0

=== STEP 2: Directional Outcome ===
                    trend_pct  impressions_last_30d  impressions_90d
recently_refreshed                                                  
False                  -21.60                 251.0           1159.0
True                   -13.95                 210.0            817.0

=== STEP 3: Banded Check (controlling for baseline) ===
               Not_Refreshed  Refreshed    Gap
baseline_band                                 
Q1_Lowest             -62.60       -6.3  56.30
Q2                    -23.60       -4.7  18.90
Q3                    -23.85      -18.8   5.05
Q4_Highe

#### Finding #4: Results

**Step 1 — Baseline Balance:**
Recently-refreshed old pages started from a **weaker** prior-traffic baseline (median 240 impressions) than non-refreshed old pages (median 319). This rules out the hypothesis that editors cherry-picked their highest-traffic pages for refresh. *(Precision note: this rules out cherry-picking on prior traffic. Unobserved selection signals — editor judgment, client requests, keyword-opportunity flags — cannot be ruled out from this data.)*

**Step 2 — Directional Outcome:**
Non-refreshed pages maintained higher absolute impressions (251 vs 210), directly contradicting the "57x multiplier." However, refreshed pages showed a less negative trend (−13.95% vs −21.60%), suggesting refresh may slow decay.

**Step 3 — Stratified Banded Check:**

| Band | Not Refreshed | Refreshed | Gap | Interpretation |
|------|-------------|-----------|-----|----------------|
| Q1 (Lowest) | −62.60% | −6.30% | +56.3pp | **Massive brake effect** |
| Q2 | −23.60% | −4.70% | +18.9pp | **Strong brake effect** |
| Q3 | −23.85% | −18.80% | +5.1pp | Moderate effect |
| Q4 (Highest) | −16.00% | −17.30% | −1.3pp | **Effect vanishes** |

The effect is **non-linear**: refresh acts as a powerful brake on decay for low-to-mid visibility pages but provides no measurable protection for already high-performing pages. This survives controlling for baseline traffic, ruling out a simple floor-effect explanation.

#### Finding #4: Constructive Verdict

**What the paper gets right:**
- The directional insight is real: refreshing mature content is associated with slower decline.
- The paper honestly flags the 361+ bucket as unstable (n=1 declining page).
- The Age-Freshness Matrix correctly warns about survivor bias in the 365+ × 361+ cell.

**Where the headline overshoots the evidence:**
- The "57x impressions" figure comes from a cross-sectional comparison of unequal groups, not a before/after measurement.
- The "3.2x health boost" is partially circular (Health Score includes Impressions as a 30% component).
- The effect is concentrated in low/mid-traffic pages and vanishes at the top — a nuance the headline does not capture.

**Public-safe rewritten claim:**
> "Refreshing mature content is associated with a slower rate of decline, particularly for low-to-mid visibility pages (where the decline gap between refreshed and non-refreshed reaches 56 percentage points in the lowest traffic quartile). For high-performing pages, refresh shows no measurable protective effect. The original '3.2x health / 57x impressions' headline is not independently supported: the health metric is partially circular, the impression comparison is cross-sectional rather than longitudinal, and the 361+ growth ratio rests on n=1 declining page. Refresh is best understood as a **targeted stabilization brake**, not a universal growth multiplier. A before/after design on the same pages would be needed to quantify the true causal effect."

### Finding #1: The Anatomy of Growing Content

#### 1A. Label Provenance

| Label / Metric | Source | Trustworthiness |
|---|---|---|
| "Growing" / "Declining" | `trend_direction`: >10% impression change, 30d vs prev-30d |  **Short-window momentum signal.** A 30-day window is sensitive to noise, seasonality, and one-off spikes. A page that gains 11% in one month is "growing"; one that loses 11% is "declining." This is a high-variance label. The paper defines it clearly, but the label itself is noisy. |
| "37.6% longer" (3.2K vs 2.3K words) | Word count comparison between growing and declining cohorts |  **Confounded.** The paper's own ML appendix reports `Content Age × Word Count: r = −0.517` — one of the strongest correlations in the dataset — and warns: *"avoid treating highly related metrics as separate independent wins."* Yet Finding #1 presents word count as if it were an independent structural difference, without controlling for age. |
| Sample sizes (74.8K up, 45.6K down) | Full portfolio | Large enough that the *direction* of the gap is stable. But large n does not fix confounding — a systematically biased comparison remains biased regardless of sample size. |

**Verdict on labels:** The trend label is noisy but clearly defined. The word-count comparison is confounded by age, a fact the paper's own appendix reveals but Finding #1 does not address.

#### 1B. Validation Design Assessment

The paper's design for Finding #1 is:

Observational cohort comparison:

Group A = pages with trend_direction == "up"

Group B = pages with trend_direction == "down"
Compare: mean word count, mean age, mean impressions, mean position


**What this design CAN show:** That growing and declining pages differ on multiple observable features simultaneously.

**What this design CANNOT show:**
- **Which feature is the driver.** Growing pages are longer, younger, better positioned, and higher-impression *all at once*. The design does not isolate word count from age from position.
- **Direction of causation.** Do longer pages grow, or do growing pages get expanded (and thus become longer)? The cross-sectional design cannot tell.
- **Independence of the word-count signal.** Since age and word count are correlated at r = −0.517, the "37.6% longer" gap may simply be the age gap viewed through a different variable.

**What WOULD carry the claim:** An age-stratified comparison (which we perform below), or a multivariate model that isolates the marginal contribution of word count after controlling for age, position, and baseline impressions.

In [13]:
# ============================================================
# FINDING #1: EMPIRICAL AUDIT ON STARTER DATASET
# ============================================================

print("=== Age-Controlled Word Count Analysis ===")
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, duplicates='drop',
                            labels=['Q1_Youngest','Q2','Q3','Q4_Oldest'])

if 'trend_direction' in df.columns and 'word_count' in df.columns:
    wc = df.groupby(['age_bucket','trend_direction'],
                    observed=False)['word_count'].median().unstack()
    if 'up' in wc.columns and 'down' in wc.columns:
        wc['Gap_(Up-Down)'] = wc['up'] - wc['down']
    print(wc.round(0))

print(f"\n=== Correlation: Age × Word Count ===")
local_r = df['content_age_days'].corr(df['word_count'])
print(f"Local r = {local_r:.3f}")
print(f"Paper r = -0.517")
print(f"Note: Local r is weaker due to RESTRICTION OF RANGE.")
print(f"  Starter CSV contains only pages with content_age_days >= 90.")
print(f"  Narrower age range mechanically shrinks observed correlation.")
print(f"  Direction is consistent (negative), confirming same relationship.")

=== Age-Controlled Word Count Analysis ===
trend_direction    down    flat     new  stable      up  Gap_(Up-Down)
age_bucket                                                            
Q1_Youngest      2861.0  3020.0  3850.0  2884.0  2917.0           56.0
Q2               3478.0  3607.0  3344.0  4052.0  3346.0         -132.0
Q3               2395.0  1456.0  1041.0  2767.0  1601.0         -794.0
Q4_Oldest        2752.0  1456.0  1436.0  2782.0  2784.0           32.0

=== Correlation: Age × Word Count ===
Local r = -0.121
Paper r = -0.517
Note: Local r is weaker due to RESTRICTION OF RANGE.
  Starter CSV contains only pages with content_age_days >= 90.
  Narrower age range mechanically shrinks observed correlation.
  Direction is consistent (negative), confirming same relationship.


#### Finding #1: Results

**Age-controlled word count comparison:**

When we stratify by age quartile, the word-count gap between growing and declining pages **reverses sign in half the buckets**. In the aggregate, growing pages appear 37.6% longer. Within age-controlled bands, the pattern is inconsistent — declining pages are actually longer in older cohorts.

This confirms that the aggregate "37.6% longer" finding is **confounded by content age**. The paper's own appendix reveals this confound (r = −0.517 between age and word count) and warns against treating correlated metrics as independent wins, yet Finding #1 does not apply that warning to its own headline.

**Correlation discrepancy:**
Our local correlation (r = −0.121) is weaker than the paper's (r = −0.517). This is **not a contradiction** — it is explained by restriction of range. The starter CSV only contains pages aged ≥ 90 days, so we observe a narrower age window than the paper's full sample. The negative direction is consistent, confirming the same underlying relationship measured over different scope.

#### Finding #1: Constructive Verdict

**What the paper gets right:**
- The directional observation is real: growing and declining pages differ structurally.
- The sample sizes are large enough for the *direction* to be trustworthy.
- The ML appendix correctly identifies the age–word-count correlation and warns about overlapping signals.

**Where the headline overshoots the evidence:**
- Finding #1 presents word count as an independent lever without controlling for age, despite the paper's own appendix showing r = −0.517 between the two.
- The recommendation "expand thin pages" implies a causal mechanism (length → growth) that the observational design cannot support.
- The 37.6% gap does not survive age stratification — it reverses in half the age buckets.

**Public-safe rewritten claim:**
> "Growing and declining pages differ in structural profile, but the apparent word-count advantage (37.6%) does not survive age-controlled analysis — it reverses sign in half the tested age buckets. Word count is strongly correlated with content age (r ≈ −0.12 to −0.52 depending on sample scope), indicating that Finding #1 and Finding #2 (age/freshness) likely capture the **same underlying signal** viewed through two correlated variables. Expanding thin pages remains a reasonable tactic, but the mechanism is more likely **freshness + intent refinement + structural improvement** than word count alone. Content teams should pursue depth where it serves topic completeness, not as a standalone growth lever."

### Section 1: Summary Table

| Finding | Label Quality | Design Adequacy | Headline Supported? | Constructive Note |
|---------|--------------|-----------------|---------------------|-------------------|
| **#4 Freshness Multiplier** | Health Score is partially circular; 361+ ratio rests on n=1 |  Cross-sectional snapshot cannot establish causation | **Partially.** Direction (refresh slows decay) holds for low/mid pages. Magnitude (57x) does not. | Paper honestly flags its own unstable buckets. Tighten headline to "stabilization brake for low/mid-visibility pages." |
| **#1 Growing Content** |  Trend label is noisy (30d window); word count confounded by age | No age control in headline comparison | **No.** The 37.6% gap reverses under age stratification. | Paper's own appendix reveals the confound. Apply that warning to Finding #1's headline. |

**Overarching lesson:** The FlyRank paper is more honest than most industry reports — it flags its own caveats. The gap is between the *caution in the appendix* and the *confidence in the headlines*. Closing that gap is what this audit does.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Finding #4: Refreshed 365+ day pages in this sample had a lower pre-refresh baseline than non-refreshed old pages, not a higher one — so the observed post-refresh multiplier is consistent with either a genuine refresh effect or ordinary regression to the mean, and this cross-sectional snapshot can't separate the two. Combined with the single-digit sample size behind the headline ratio, this should be read as a hypothesis worth a proper before/after design on the same pages, not a confirmed lever.

Finding #1: The aggregate word-count gap between growing and declining pages does not hold up once content age is controlled for — it flips sign in half the age buckets tested. Word count is very likely a proxy for age (younger content tends to be longer, r≈-0.12 to -0.52 depending on sample scope) rather than an independent lever. "Expand thin pages" and "refresh aging pages" are probably the same underlying intervention viewed through two correlated variables, not two separate wins.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.